# 因子测试 · 第一步：读取 y 与股债/波动率等指标

1. 读取 **大盘涨跌-小盘涨跌**：优先从 factor_build/outputs/01_y_timeseries.xlsx，否则从 Data 行情重建（pipeline 通过读取上一层级 Excel 进行下一步）
2. 读取 **股债 Excel**（config.IMPLICATION_EXCEL_FILENAME）的**所有 sheet**
3. 将股债 Excel 中各指标与 **大盘涨跌-小盘涨跌** 按 交易日期 对齐，得到一张总表并**输出 Excel**，供后续步骤调用

In [19]:
import os
import sys
import pandas as pd
import numpy as np

_cwd = os.getcwd()
_root = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'factor_test' else _cwd
if _root not in sys.path:
    sys.path.insert(0, _root)
import config

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x)

DATA_DIR = config.get_data_dir()
DATE_COL = config.DATE_COL
# 本层读取：factor_build 输出
OUTPUTS_DIR = os.path.join(_root, 'factor_build', getattr(config, 'FACTOR_BUILD_OUTPUTS', 'outputs'))
# 本层输出：factor_test 输出，供后续步骤读取
FACTOR_TEST_OUT = os.path.join(_root, 'factor_test', getattr(config, 'FACTOR_TEST_OUTPUTS', 'outputs'))
Y_COL_NAME = "大盘涨跌-小盘涨跌"

## 1. 读取 大盘涨跌-小盘涨跌（来自 factor_build 输出）

In [20]:
path_y = os.path.join(OUTPUTS_DIR, '01_y_timeseries.xlsx')
if os.path.isfile(path_y):
    df_y = pd.read_excel(path_y)
    df_y[DATE_COL] = pd.to_datetime(df_y[DATE_COL]).dt.date
    if 'y' in df_y.columns:
        df_y = df_y.rename(columns={'y': Y_COL_NAME})
    elif '涨跌幅_50' in df_y.columns and '涨跌幅_1000' in df_y.columns:
        df_y[Y_COL_NAME] = df_y['涨跌幅_50'].astype(float) - df_y['涨跌幅_1000'].astype(float)
    else:
        raise KeyError('01_y_timeseries.xlsx 中需有 y 或 涨跌幅_50/涨跌幅_1000')
    df_y = df_y[[DATE_COL, Y_COL_NAME]].dropna()
    print('已从 factor_build 输出读取:', path_y)
else:
    all_files = [f for f in os.listdir(DATA_DIR) if f.endswith('.xlsx') and not f.startswith(config.SKIP_FILE_PREFIX)]
    file_50 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI50)), None)
    file_1000 = next((f for f in all_files if all(k in f for k in config.FILE_KEYWORDS_CSI1000)), None)
    if not file_50 or not file_1000:
        raise FileNotFoundError('未找到 01_y_timeseries.xlsx 且 Data 下无行情文件')
    df50 = pd.read_excel(os.path.join(DATA_DIR, file_50))[[DATE_COL, config.PCT_COL]].dropna()
    df1000 = pd.read_excel(os.path.join(DATA_DIR, file_1000))[[DATE_COL, config.PCT_COL]].dropna()
    df50[DATE_COL] = pd.to_datetime(df50[DATE_COL]).dt.date
    df1000[DATE_COL] = pd.to_datetime(df1000[DATE_COL]).dt.date
    df50 = df50.rename(columns={config.PCT_COL: '涨跌幅_50'})
    df1000 = df1000.rename(columns={config.PCT_COL: '涨跌幅_1000'})
    df_y = df50.merge(df1000, on=DATE_COL, how='inner')
    df_y = df_y.dropna().loc[(df_y['涨跌幅_50'] != 0) & (df_y['涨跌幅_1000'] != 0)]
    df_y[Y_COL_NAME] = df_y['涨跌幅_50'].astype(float) - df_y['涨跌幅_1000'].astype(float)
    df_y = df_y[[DATE_COL, Y_COL_NAME]].reset_index(drop=True)
    if getattr(config, 'CUTOFF_DATE', None):
        import datetime
        cut = config.CUTOFF_DATE
        if isinstance(cut, (tuple, list)) and len(cut) >= 3:
            cut = datetime.date(int(cut[0]), int(cut[1]), int(cut[2]))
        df_y = df_y[df_y[DATE_COL] <= cut]
    print('已从 Data 行情重建 大盘涨跌-小盘涨跌')
print('df_y 行数:', len(df_y))
display(df_y.head(5))

已从 factor_build 输出读取: /Users/zhanghongyi/Desktop/Github/VINS Project/b_program/p3_adjusted_program/factor_build/outputs/01_y_timeseries.xlsx
df_y 行数: 2406


,交易日期,大盘涨跌-小盘涨跌
0,2015-11-19,-1.90
1,2015-11-20,-2.07
2,2015-11-23,0.55
3,2015-11-24,-1.50
4,2015-11-25,-1.78


## 2. 读取股债 Excel 全部 sheet

In [21]:
impl_file = getattr(config, 'IMPLICATION_EXCEL_FILENAME', '数据_波动率与股债指数.xlsx')
impl_path = os.path.join(DATA_DIR, impl_file)
if not os.path.isfile(impl_path):
    raise FileNotFoundError('未找到股债/波动率 Excel: ' + impl_path)

sheets_impl = pd.read_excel(impl_path, sheet_name=None)
print('股债 Excel sheet 数:', len(sheets_impl))
print('Sheet 名:', list(sheets_impl.keys()))

股债 Excel sheet 数: 5
Sheet 名: [' 国债收益率', '汇率收盘价', '中国波动率收盘价', '中证股债风险平均指数收盘价', '大宗商品收盘价']


## 3. 规整各 sheet：统一日期列，取数值列

In [22]:
def normalize_date_col(df):
    df = df.copy()
    for c in ['交易日期', 'Date', '日期', 'date']:
        if c in df.columns:
            if c != DATE_COL:
                df = df.rename(columns={c: DATE_COL})
            df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
            break
    if DATE_COL not in df.columns and len(df.columns) >= 1:
        df = df.rename(columns={df.columns[0]: DATE_COL})
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
    return df

def get_value_columns(df, sheet_name):
    """除日期列外的列视为数值指标，列名过长时用 sheet 名代替。"""
    cols = [c for c in df.columns if c != DATE_COL and str(c).strip() != '']
    return cols

impl_tables = {}
for name, df in sheets_impl.items():
    df = normalize_date_col(df)
    if DATE_COL not in df.columns:
        continue
    value_cols = get_value_columns(df, name)
    for c in value_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    impl_tables[name] = df[[DATE_COL] + value_cols].copy()

print('规整后 sheet 数:', len(impl_tables))
for name, df in impl_tables.items():
    print(f'  {name}: {list(df.columns)}')

规整后 sheet 数: 5
   国债收益率: ['交易日期', 'Unnamed: 1', 'Unnamed: 2']
  汇率收盘价: ['交易日期', datetime.datetime(2019, 1, 1, 0, 0), 'Unnamed: 2']
  中国波动率收盘价: ['交易日期', datetime.datetime(2019, 1, 1, 0, 0)]
  中证股债风险平均指数收盘价: ['交易日期', datetime.datetime(2019, 1, 1, 0, 0)]
  大宗商品收盘价: ['交易日期', datetime.datetime(2019, 1, 1, 0, 0), 'Unnamed: 2']


/var/folders/nw/qn0916dd3g12_8kqhgfzzgp40000gn/T/ipykernel_50211/798748591.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
/var/folders/nw/qn0916dd3g12_8kqhgfzzgp40000gn/T/ipykernel_50211/798748591.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce').dt.date
/var/folders/nw/qn0916dd3g12_8kqhgfzzgp40000gn/T/ipykernel_50211/798748591.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors='coerce

## 4. 与 大盘涨跌-小盘涨跌 按 交易日期 合并为一张总表

In [23]:
merged = df_y[[DATE_COL, Y_COL_NAME]].copy()
for name, df in impl_tables.items():
    value_cols = [c for c in df.columns if c != DATE_COL]
    if not value_cols:
        continue
    # 多列时用 sheet名_列名 避免重名
    rename = {c: f'{name}_{c}' if len(value_cols) > 1 else name for c in value_cols}
    df_merge = df[[DATE_COL] + value_cols].rename(columns=rename)
    merged = merged.merge(df_merge, on=DATE_COL, how='left')

merged = merged.sort_values(DATE_COL).reset_index(drop=True)
df_indicators = merged
print('合并后总表形状:', df_indicators.shape)
print('列:', list(df_indicators.columns))
display(df_indicators.sample(10))

合并后总表形状: (2406, 2)
列: ['大盘涨跌-小盘涨跌', '中国波动率收盘价']


,大盘涨跌-小盘涨跌,中国波动率收盘价
2225,-1.91,17.2555
2347,-1.90,17.1213
1446,0.44,18.0403
1746,-1.28,18.3167
491,4.43,NaN
1069,2.21,18.1028
971,-0.06,13.4182
1261,1.13,23.9516
1846,0.21,14.7827
2299,0.53,17.1213


## 5. 输出 Excel 与汇总

- **df_y**：大盘涨跌-小盘涨跌 时序（来自 factor_build 输出）
- **df_indicators**：大盘涨跌-小盘涨跌 + 股债 Excel 全部指标按日期对齐
- **输出**：写入 `factor_test/outputs/01_y_and_implication.xlsx`，后续步骤通过读取此文件进行下一步

In [24]:
print('日期范围:', df_indicators[DATE_COL].min(), '~', df_indicators[DATE_COL].max())
print('各列非空数:')
print(df_indicators.notna().sum())

os.makedirs(FACTOR_TEST_OUT, exist_ok=True)
out_path = os.path.join(FACTOR_TEST_OUT, '01_y_and_implication.xlsx')
df_indicators.to_excel(out_path, index=False)
print('已写出:', out_path)

KeyError: '交易日期'